# Simplified Transform Analysis
## Effect of Measurement Noise on Transform Stability

It was confirmed by expermental data analysis that measurement introduces 
some noise to the values of instrumental magnitudes. For the current setup
it's typically 50-100 mmag.


Let's set up an experiment and geterate a test sequence

In [ ]:
from vsopy.util import ValErr
from experiment import Experiment

SEED = 42


EXPERIMENT = Experiment(SEED)

SEQUENCE = EXPERIMENT.generate_sequence(
    Experiment.GenParams(
        num_stars=20,
        mag=(10.0, 15.0),
        ci=(-0.5, 1.0),
        uncert=ValErr(0.05, 0.05)))

Let's select a list of values for measurement noise.  For each value
measure the sequence N=200 times and calculate transform eacchh time.
Calculate average value of the transform, stdev of treansform value,
and mean uncertainty of the transform value

In [ ]:
from measurement import SimpleMeasurement
from util import simple_transform_stats
from astropy.table import QTable


SCATTER_VALS = [.01, .012, .015, .02, .025, .03, .04, .05, .06, .075, .1, .12, .15, .2, .25, .3, .4, .5]
TR_STD_BY_SCATTER = QTable(
    [ simple_transform_stats(
        EXPERIMENT.create_transforms(
            SEQUENCE,
            EXPERIMENT.create_measurement(
                SimpleMeasurement,
                params = SimpleMeasurement.Params(
                    offset=1.0,
                    scatter=s,
                    uncert=ValErr(0.03, 0.03))),
                num_measurements=200))
               | {'scatter': s} for s in SCATTER_VALS ]
    )
TR_STD_BY_SCATTER.to_pandas()

Plot the data

In [ ]:
from plotnine import ggplot, aes, geom_line, geom_point, labs, theme, scale_x_log10, facet_grid, facet_wrap

(ggplot(TR_STD_BY_SCATTER.to_pandas().melt(id_vars=['scatter'], var_name='transform', value_name='stdev'))
    + aes(x='scatter', y='stdev') + geom_point()
 #+ facet_grid('transform ~ .', scales='free_y') +
    + facet_wrap('~transform', scales='free_y')
    + scale_x_log10()
    + theme(figure_size=(10, 10)))

When offset is the same for both bands (no color distortions), 
$$T_{ab} \approx 1$$
$$T_{a} \approx 0$$
$$T_{b} \approx 0$$

The chart above shows that measuement error above .2 mag causes $T_{ab}$
instability and better be under .1 mag (that's realistic)